## Usage example

Here is an example of a LSA pipeline that:
1. Ingests a collection of texts
2. Makes the corresponding document-term matrix using stemming and removing stop words
3. Extracts 40 topics
4. Shows a table with the extracted topics
5. Shows a table with statistical thesaurus entries for selected words  

In [ ]:
# Packages used
use ML::LatentSemanticAnalyzer;
use ML::LatentSemanticAnalyzer::Utilities;
use Lingua::EN::Stem::Porter;
use Data::Reshapers;
use Statistics::OutlierIdentifiers;

In [ ]:
# Collection of texts
my @dsAbstracts = ML::LatentSemanticAnalyzer::Utilities::get-abstracts-dataset();

# Remove non-strings
@dsAbstracts .= grep({ $_<Abstract> ~~ Str:D });

say "@dsAbstracts.elems : {@dsAbstracts.elems}";
my %docs = @dsAbstracts.map(*<ID>) Z=> @dsAbstracts.map(*<Abstract>);
say "%docs.elems : {%docs.elems}";

In [ ]:
#% html
@dsAbstracts.pick(6)
==> to-html(field-names => <ID Name Title Abstract Track>)

In [ ]:
# Stemmer function (to preprocess words in the pipeline below)
porter("preparation")

In [ ]:
# Derive stemming rules (to be used in the LSA pipeline)
my %stemming-rules = %docs.values.join(' ').lc.split(/\s | <:punct> /, :skip-empty)>>.trim.unique.map({ $_ => porter($_) });
deduce-type(%stemming-rules)

In [ ]:
# Words to show statistical thesaurus entries for
sink my @words = <notebook computational function neural talk programming>;

In [ ]:
# Reproducible results (within a single session)
sink srand(12);

In [ ]:
# Echo function used in the pipeline
my &echo-function = {.say for |$_};

In [ ]:
# LSA pipeline
my $lsaObj =
        ML::LatentSemanticAnalyzer.new
                .make-document-term-matrix(docs => %docs, :stop-words, :%stemming-rules, :3min-length)
                .apply-term-weight-functions(
                    global-weight-func => "IDF", 
                    local-weight-func => "None", 
                    normalizer-func => "Cosine"
                )
                .extract-topics(:40number-of-topics, :10min-number-of-documents-per-term, method => "SVD", :60max-steps, tolerance => 1e-4)
                .echo-topics-interpretation(:12number-of-terms, :dataset, :wide-form, :&echo-function)
                .echo-statistical-thesaurus(
                    terms => @words.map({ porter($_) }), 
                    :wide-form, 
                    :12number-of-nearest-neighbors, 
                    method => "cosine", 
                    :&echo-function
                );


In [ ]:
$lsaObj.echo-document-term-matrix-statistics()

---

## Find outliers in the topics interpretation data frame

In [ ]:
my @dsTopicsLongForm = |$lsaObj.get-topics-interpretation(:120number-of-terms, :dataset, :!wide-form, :!echo).take-value;
deduce-type(@dsTopicsLongForm)

In [ ]:
#% html
@dsTopicsLongForm.pick(12)
==> to-html(field-names => <Topic Term Score>)

In [ ]:
# Group by "Topic" and select rows where Score is an outlier according to your function
my @dfTopicsOfOutliersLongForm =
    group-by(@dsTopicsLongForm, "Topic").kv
    .map(-> $t, @g { @g[outlier-identifier(@g.map(*<Score>), identifier => &top-outliers o &quartile-identifier-parameters)] })
    .flat(1);

deduce-type(@dfTopicsOfOutliersLongForm)

In [ ]:
#% html
@dfTopicsOfOutliersLongForm.pick(12)
==> to-html(field-names => <Topic Term Score>)

In [ ]:
group-by(@dfTopicsOfOutliersLongForm, "Topic")».elems.sort(*.value).reverse

----

## Representation of texts

In [ ]:
sink my $query = q:to/END/;
Wolfram notebooks have been copied by other computational systems.
END

In [ ]:
$lsaObj.represent-by-terms($query, :!apply-lsi-functions).take-value

In [ ]:
my $m2 = $lsaObj.represent-by-topics($query, :apply-lsi-functions).take-value;
say $m2;
$m2.transpose.print

----

## Profiling

In [ ]:
# LSA pipeline
my $lsaObj = ML::LatentSemanticAnalyzer.new;

In [ ]:
$lsaObj .= make-document-term-matrix(docs => %docs, :stop-words, :%stemming-rules, :3min-length);

In [ ]:
$lsaObj .= apply-term-weight-functions(
                global-weight-func => "IDF", 
                local-weight-func => "None", 
                normalizer-func => "Cosine"
           )        

In [ ]:
$lsaObj .= extract-topics(:40number-of-topics, :10min-number-of-documents-per-term, method => "SVD", :60max-steps, tolerance => 1e-4)

In [ ]:
$lsaObj .= echo-topics-interpretation(:12number-of-terms, :dataset, :wide-form, :&echo-function)

In [ ]:
$lsaObj .= echo-statistical-thesaurus(
                terms => @words.map({ porter($_) }), 
                :wide-form, 
                :12number-of-nearest-neighbors, 
                method => "cosine", 
                :&echo-function
            );
